# Notebook 00 — Open, Inspect, and Prepare Revelation SVG Text Units

This notebook is the intake and preprocessing notebook for the Revelation SVG translation workflow. It opens the Revelation SVG source files, extracts translatable text units, inspects the resulting text-unit dataframe, and prepares a JSON checkpoint for the translation notebook.

This workflow is based on the earlier SVG translation notebooks used for the BST and TBE examples, but this Revelation version includes several bespoke preprocessing rules that are specific to these graphics. The goal is not to create a fully general SVG translation engine, but to prepare this particular set of Revelation graphics carefully enough that the translation notebook can focus on translating only the text that should actually be translated.

## Illustrator SVG export settings

The Revelation source SVG files should be exported from Adobe Illustrator with browser-compatible SVG structure rather than Illustrator-private editing data. In Illustrator:

1. Save a copy and choose the SVG file format.
2. Use `SVG 1.1`, `Only Glyphs Used`, and `Presentation Attributes`.
3. Importantly: do ***not*** check `Preserve Illustrator Editing Capabilities`.
4. Check `Output fewer tspans`, `Use textpath`, and `Responsive`.

Leaving `Preserve Illustrator Editing Capabilities` unchecked avoids embedded Illustrator-private structures that can be altered by XML parse/write serialization and then fail to reopen correctly in Illustrator after reconstruction.

![Illustrator SVG export settings](Illustrator_svg_settings.jpg)

## What this notebook does

1. Loads the Revelation SVG source files.
2. Extracts SVG text elements and text spans into a dataframe of translation units.
3. Preserves metadata needed for reconstruction, including source file, group stack, element path, text ID, tspan ID, and tspan index.
4. Applies Revelation-specific filtering and classification rules before translation.
5. Creates a filtered `df_translate` dataframe containing only rows that should be sent to the LLM.
6. Saves a JSON checkpoint for use in the translation notebook.

## Revelation-specific preprocessing

This notebook includes several custom rules added for the Revelation graphics:

- Greek text is detected and preserved unchanged.
- Bible-reference-only strings are detected and preserved unchanged.
- Cross-reference strings beginning with `cf.` are preserved unchanged.
- Verse excerpts that begin with chapter:verse markers remain translatable.
- Mixed text/reference strings remain translatable.
- Symbol-only text such as decorative markers and asterisks is preserved unchanged.
- Repeated phrase patterns such as `cross-reference(s)` and `Revelation` are inspected and annotated with language-neutral notes for later prompt handling.

## Notes on Bible references

The Revelation graphics contain many Bible references in varied formats, including abbreviated book names, full book names, chapter-only references, chapter:verse references, ranges, asterisks, and cross-reference lists. This notebook uses a helper file, `bible_reference_helpers.py`, to support reference detection through a reusable Bible book abbreviation map.

At this stage, the workflow does not attempt to normalize Bible abbreviations into a target language. Instead, it distinguishes between references that should be preserved and text that should still be translated.

## Bespoke workflow note

This notebook is intentionally somewhat bespoke. The Revelation graphics contain specialized visual structures, repeated labels, Greek words, cross-reference lists, and structural notation that require project-specific handling. Some structural strings, such as chiasm labels and outline markers, are left for the translation model to handle during the translation stage rather than being preprocessed here.

Several modifications in this notebook were made with assistance from Codex inside VS Code. The changes were kept localized and incremental so that each preprocessing rule could be inspected before moving on to the translation stage.

In [1]:
# Set directories

from pathlib import Path

PROJECT_ROOT = Path.cwd()
SVG_SOURCE_DIR = PROJECT_ROOT / "svg_source_files"
SVG_OUTPUT_DIR = PROJECT_ROOT / "svg_output_files"
JSON_DIR = PROJECT_ROOT / "json_files"

# Source/input folder should already exist
assert SVG_SOURCE_DIR.exists(), f"Missing folder: {SVG_SOURCE_DIR}"

# Output folders can be created automatically
SVG_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JSON_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT.name)
print("SVG_SOURCE_DIR:", SVG_SOURCE_DIR.relative_to(PROJECT_ROOT.parent))
print("JSON_DIR:", JSON_DIR.relative_to(PROJECT_ROOT.parent))
print("SVG_OUTPUT_DIR:", SVG_OUTPUT_DIR.relative_to(PROJECT_ROOT.parent))

PROJECT_ROOT: rev
SVG_SOURCE_DIR: rev\svg_source_files
JSON_DIR: rev\json_files
SVG_OUTPUT_DIR: rev\svg_output_files


In [2]:
# Display svg files in source directory

svg_paths = sorted(SVG_SOURCE_DIR.glob("*.svg"))

print("Found SVG files:", len(svg_paths))
for p in svg_paths:
    print(" -", p.name)


Found SVG files: 1
 - StructureOfRevelation.svg


## Read and parse SVG files
**NOTE:** This workflow will process *all* svg files contained in the svg_source_files directory 

In [3]:
from lxml import etree
import pandas as pd
import json
import re

SVG_NS = "http://www.w3.org/2000/svg"
NS = {"svg": SVG_NS}

def localname(tag: str) -> str:
    return tag.split("}", 1)[1] if tag and tag.startswith("{") else (tag or "")

def norm_ws(s: str) -> str:
    if s is None:
        return ""
    return re.sub(r"\s+", " ", s).strip()

def parse_style(style: str):
    out = {}
    if not style:
        return out
    for part in style.split(";"):
        part = part.strip()
        if not part or ":" not in part:
            continue
        k, v = part.split(":", 1)
        out[k.strip()] = v.strip()
    return out

def group_label(g) -> str:
    return g.get("data-name") or g.get("id") or "g"

def compute_group_stack(el) -> list[str]:
    stack = []
    cur = el.getparent()
    while cur is not None:
        if localname(cur.tag) == "g":
            stack.append(group_label(cur))
        cur = cur.getparent()
    return list(reversed(stack))

def element_path(el) -> str:
    parts = []
    cur = el
    while cur is not None:
        tag = localname(cur.tag)
        if tag in ("svg", "g", "text", "tspan"):
            _id = cur.get("id")
            if _id:
                parts.append(f"{tag}#{_id}")
            else:
                parent = cur.getparent()
                if parent is None:
                    parts.append(tag)
                else:
                    same = [
                        c for c in parent
                        if hasattr(c, "tag") and localname(c.tag) == tag
                    ]
                    idx = same.index(cur) + 1  # 1-based index
                    parts.append(f"{tag}[{idx}]")
        cur = cur.getparent()
    return "/".join(reversed(parts))

def get_text_and_tspans(text_el):
    tspans = []
    chunks = []
    if text_el.text not in (None, ""):
        chunks.append(text_el.text)

    for t in text_el.findall(".//svg:tspan", namespaces=NS):
        t_text = t.text or ""
        if t_text != "":
            chunks.append(t_text)
        tspans.append({
            "tspan_id": t.get("id"),
            "tspan_text": t_text,
            "tspan_text_norm": norm_ws(t_text),
            "x": t.get("x"),
            "y": t.get("y"),
            "dx": t.get("dx"),
            "dy": t.get("dy"),
            "style": t.get("style"),
            "class": t.get("class"),
        })
        if t.tail not in (None, ""):
            chunks.append(t.tail)

    return "".join(chunks), tspans

def extract_text_inventory(svg_path: Path) -> pd.DataFrame:
    parser = etree.XMLParser(remove_blank_text=False, recover=True, huge_tree=True)
    tree = etree.parse(str(svg_path), parser)
    root = tree.getroot()

    rows = []
    for text_el in root.xpath("//svg:text", namespaces=NS):
        full_text, tspans = get_text_and_tspans(text_el)
        full_text_norm = norm_ws(full_text)
        if not full_text_norm:
            continue

        gstack = compute_group_stack(text_el)
        style = parse_style(text_el.get("style"))

        rows.append({
            "source_file": svg_path.name,
            "text_id": text_el.get("id"),
            "group_stack": " / ".join(gstack),
            "group_depth": len(gstack),
            "text_raw": full_text,
            "text_norm": full_text_norm,
            "has_tspans": len(tspans) > 0,
            "tspan_count": len(tspans),
            "x": text_el.get("x"),
            "y": text_el.get("y"),
            "transform": text_el.get("transform"),
            "class": text_el.get("class"),
            "style": text_el.get("style"),
            "style_font_family": style.get("font-family"),
            "style_font_size": style.get("font-size"),
            "style_text_anchor": style.get("text-anchor"),
            "element_path": element_path(text_el),
            "tspans": tspans,
        })

    df = pd.DataFrame(rows).sort_values(["group_stack", "element_path"]).reset_index(drop=True)
    return df

all_dfs = []
for p in svg_paths:
    df = extract_text_inventory(p)
    all_dfs.append(df)

    out_json = JSON_DIR / f"{p.stem}.text_extract.json"
    records = df.to_dict(orient="records")
    out_json.write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"{p.name}: extracted {len(df)} text elements -> {out_json.name}")

df_all = pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()
print("Total extracted across files:", len(df_all))

missing_space_pattern_rows = df_all[df_all["text_norm"].str.contains(r"[a-z][A-Z]", regex=True, na=False)]
print("Likely missing-space pattern rows:", len(missing_space_pattern_rows))
missing_space_pattern_rows[["text_raw", "text_norm", "group_stack", "element_path"]]


StructureOfRevelation.svg: extracted 249 text elements -> StructureOfRevelation.text_extract.json
Total extracted across files: 249
Likely missing-space pattern rows: 0


,text_raw,text_norm,group_stack,element_path


### Human-readable summary

In [4]:
def outline(df: pd.DataFrame, max_chars: int = 90):
    for i, r in df.iterrows():
        snippet = r["text_norm"]
        if len(snippet) > max_chars:
            snippet = snippet[:max_chars-1] + "…"
        print(f"[{i:04d}] {r['group_stack']} :: {snippet}")

for fname in df_all["source_file"].unique():
    print("\n" + "="*80)
    print(fname)
    print("="*80)
    outline(df_all[df_all["source_file"] == fname].reset_index(drop=True), max_chars=110)



StructureOfRevelation.svg
[0000] _x37__bowls :: cf. Ex 9:9-11
[0001] _x37__bowls :: cf. Ex 7:17-21,Is 49:26,Rev 8:8-10,11:6
[0002] _x37__bowls :: cf. Rev 14:18
[0003] _x37__bowls :: cf. Rev 9:14,13:1
[0004] _x37__bowls :: 232 cross-references
[0005] _x37__bowls :: All life insea dies
[0006] _x37__bowls :: Rev 16:3
[0007] _x37__bowls :: Water toblood
[0008] _x37__bowls :: Rev 16:4-7
[0009] _x37__bowls :: Darkness
[0010] _x37__bowls :: 7 Bowls
[0011] _x37__bowls :: Rev 16:10-11
[0012] _x37__bowls :: Revelation 16:1-16
[0013] _x37__bowls :: Sores
[0014] _x37__bowls :: Fire from Sun
[0015] _x37__bowls :: Euphratesdries
[0016] _x37__bowls :: cf. Joe 2,Rev 6:12
[0017] _x37__bowls :: Rev 16:2
[0018] _x37__bowls :: Rev 16:8-9
[0019] _x37__bowls :: Rev 16:12-14
[0020] _x37__churches :: 7 Churches
[0021] _x37__churches :: Revelation 1-3
[0022] _x37__churches :: see 7 Churches of Revelation graphic
[0023] _x37__churches :: cf. Is 41:4 & 44:6 & 48:12, Dan 7:13, Zec 4:2 & 12:10, Mt 24:30 & 26:64, 

### Create LLM-ready table of elements

In [5]:
import hashlib
import pandas as pd

def make_unit_key(source_file: str, element_path: str, text_id: str | None, source_text: str) -> str:
    # Hash stable text-level information.
    parts = [source_file, element_path]
    if text_id is not None and not pd.isna(text_id) and str(text_id).strip():
        parts.append(f"text_id={text_id}")
    parts.append(f"source_text={source_text}")
    raw = "|".join(parts)
    return hashlib.sha1(raw.encode("utf-8")).hexdigest()

def nonempty_text(value) -> str:
    if value is None or pd.isna(value):
        return ""
    return str(value).strip()

def build_translation_df(df_text: pd.DataFrame) -> pd.DataFrame:
    out = []
    for _, r in df_text.iterrows():
        source_file = r["source_file"]
        element_path = r["element_path"]
        group_stack = r["group_stack"]
        text_id = r.get("text_id")

        src = nonempty_text(r.get("text_norm"))
        if not src:
            tspans = r.get("tspans")
            if not isinstance(tspans, list):
                tspans = []
            src = "".join(nonempty_text(t.get("tspan_text_norm")) for t in tspans)
            src = " ".join(src.split())

        if not src:
            continue

        unit_key = make_unit_key(source_file, element_path, text_id, src)
        out.append({
            "unit_key": unit_key,
            "unit_type": "text",
            "source_file": source_file,
            "group_stack": group_stack,
            "element_path": element_path,
            "text_id": text_id,
            "tspan_id": None,
            "tspan_idx": None,
            "source_text": src,
        })

    columns = ["unit_key", "unit_type", "source_file", "group_stack", "element_path", "text_id", "tspan_id", "tspan_idx", "source_text"]
    df_units = pd.DataFrame(out, columns=columns)
    # helpful sorting for review
    return df_units.sort_values(["source_file", "group_stack", "element_path", "unit_type", "tspan_idx"]).reset_index(drop=True)

df_units = build_translation_df(df_all)
print("Total translation units:", len(df_units))
print("Rows containing Glory:", df_units["source_text"].str.contains("Glory", na=False).sum())
print("First 20 rows of df_units:")
df_units.head(20)


Total translation units: 249
Rows containing Glory: 2
First 20 rows of df_units:


,unit_key,unit_type,source_file,group_stack,element_path,text_id,tspan_id,tspan_idx,source_text
0,65971727bf7e86f3fecbb33b07be3acc2892de35,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[10],None,None,None,cf. Ex 9:9-11
1,8140f9ef3ce81a748de1998b176b8d1c1c072cd5,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[11],None,None,None,"cf. Ex 7:17-21,Is 49:26,Rev 8:8-10,11:6"
2,69c90ab68fd7372766dc12d6f01b35d3857127b6,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[12],None,None,None,cf. Rev 14:18
3,3bf7cce880b6e43304712659ba23c60caf957731,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[13],None,None,None,"cf. Rev 9:14,13:1"
4,4041ecaccb42dee3d7f29c75e00b137f86bf10cc,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[14],None,None,None,232 cross-references
5,6bb999b7f7c222c530baaa3de7036f1e146d9037,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[15],None,None,None,All life insea dies
6,3d375b2762a52fb3217330358fc98ebcbc81aacb,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[16],None,None,None,Rev 16:3
7,2e46ca8ff13d19e17a83d5683f948b0b18842b2d,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[17],None,None,None,Water toblood
8,2e8bdd1d58d4eb4202881730f7fffe13fd88c730,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[18],None,None,None,Rev 16:4-7
9,fc6ae5198a7f6b72c6c870f6aae0f115be396976,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[19],None,None,None,Darkness


## Revelation-specific text filtering: preserve Greek text


In [6]:
import unicodedata

def is_greek_text(value, threshold: float = 0.60) -> bool:
    """Return True when more than threshold of Unicode letters are Greek."""
    text = "" if value is None else str(value).strip()
    if not text:
        return False

    letters = [ch for ch in text if unicodedata.category(ch).startswith("L")]
    if not letters:
        return False

    greek_letters = [
        ch for ch in letters
        if "GREEK" in unicodedata.name(ch, "")
    ]
    return (len(greek_letters) / len(letters)) > threshold

if "translation_action" not in df_units.columns:
    df_units["translation_action"] = "translate"
else:
    action = df_units["translation_action"]
    df_units["translation_action"] = action.mask(action.isna() | action.astype(str).str.strip().eq(""), "translate")

if "translation_note" not in df_units.columns:
    df_units["translation_note"] = ""
else:
    df_units["translation_note"] = df_units["translation_note"].fillna("")

if "target_text" not in df_units.columns:
    df_units["target_text"] = None

greek_mask = df_units["source_text"].map(is_greek_text)

df_units.loc[greek_mask, "translation_action"] = "preserve"
df_units.loc[greek_mask, "target_text"] = df_units.loc[greek_mask, "source_text"]
df_units.loc[greek_mask, "translation_note"] = "Greek text preserved unchanged."

df_translate = df_units[df_units["translation_action"].eq("translate")].copy()

print("Total rows in df_units:", len(df_units))
print("Greek rows detected:", int(greek_mask.sum()))
print("Rows now marked preserve:", int(df_units["translation_action"].eq("preserve").sum()))
print("Rows still marked translate:", len(df_translate))

review_cols = [
    "source_file",
    "group_stack",
    "text_id",
    "tspan_id",
    "tspan_idx",
    "source_text",
    "translation_action",
    "translation_note",
]

df_units.loc[greek_mask, review_cols].copy()


Total rows in df_units: 249
Greek rows detected: 0
Rows now marked preserve: 0
Rows still marked translate: 249


,source_file,group_stack,text_id,tspan_id,tspan_idx,source_text,translation_action,translation_note


## Revelation-specific text filtering: detect Bible references


In [7]:
import re
from IPython.display import display

try:
    import bible_reference_helpers as brh
    bible_book_aliases = brh.get_all_reference_aliases()
    print("Loaded Bible reference aliases from bible_reference_helpers:", len(bible_book_aliases))
except ImportError as exc:
    print("Could not import bible_reference_helpers; using local fallback aliases.")
    print("Import error:", exc)
    bible_book_aliases = [
        "Gen", "Genesis", "Ex", "Exodus", "Is", "Isa", "Isaiah", "Jer", "Jeremiah",
        "Eze", "Ezek", "Ezekiel", "Dan", "Daniel", "Mal", "Malachi", "Jn", "John",
        "Rev", "Revelation", "Ap", "Apocalipsis",
    ]

def alias_to_regex(alias: str) -> str:
    return re.escape(alias).replace(r"\ ", r"\s+")

alias_pattern = "|".join(alias_to_regex(alias) for alias in sorted(set(bible_book_aliases), key=len, reverse=True))

# Matches book alias + chapter, optional :verse, optional verse range, with optional leading cf/cf.
bible_reference_re = re.compile(
    rf"(?<!\w)(?:cf\.?\s+)?(?:{alias_pattern})(?!\w)\s+\d{{1,3}}(?:\s*:\s*\d{{1,3}}(?:\s*[-–]\s*\d{{1,3}})?)?",
    flags=re.IGNORECASE,
)

verse_excerpt_re = re.compile(
    r"^\s*\d{1,3}\s*:\s*\d{1,3}(?:\s*[-–]\s*\d{1,3})?\s+\S"
)

cf_lead_re = re.compile(r"^\s*cf\.?\b", flags=re.IGNORECASE)

def starts_with_cf(text):
    """
    Return True when text begins with 'cf.' or 'cf' after stripping leading whitespace.
    This is used in the Revelation workflow because these entries are cross-reference labels.
    """
    if text is None:
        return False
    try:
        if text != text:  # NaN is not equal to itself.
            return False
    except TypeError:
        pass
    return bool(cf_lead_re.match(str(text).lstrip()))

for col, default in [
    ("translation_action", "translate"),
    ("translation_note", ""),
    ("target_text", None),
    ("contains_bible_reference", False),
    ("bible_reference_category", None),
    ("bible_reference_note", ""),
]:
    if col not in df_units.columns:
        df_units[col] = default

translation_action = df_units["translation_action"]
df_units["translation_action"] = translation_action.mask(
    translation_action.isna() | translation_action.astype(str).str.strip().eq(""),
    "translate",
)
df_units["translation_note"] = df_units["translation_note"].fillna("")
df_units["bible_reference_note"] = df_units["bible_reference_note"].fillna("")

def is_mostly_reference_text(text: str, matches: list[re.Match]) -> bool:
    remainder = str(text)
    for match in reversed(matches):
        remainder = remainder[:match.start()] + remainder[match.end():]
    allowed_annotation_chars = r"[\s,;:()\[\].\-*†‡–]+"
    remainder = re.sub(rf"^{allowed_annotation_chars}|{allowed_annotation_chars}$", "", remainder)
    remainder = re.sub(allowed_annotation_chars, "", remainder)
    return remainder == ""

def classify_bible_reference_text(value):
    text = "" if value is None else str(value).strip()
    if not text:
        return False, None, ""

    verse_excerpt = bool(verse_excerpt_re.search(text))
    matches = list(bible_reference_re.finditer(text))

    if verse_excerpt:
        return True, "leading_reference_with_text", "Leading reference with text detected; keep for translation."

    if not matches:
        return False, None, ""

    if is_mostly_reference_text(text, matches):
        if cf_lead_re.match(text):
            return True, "cross_reference_only", "Cross-reference-only Bible reference detected."
        return True, "reference_only", "Reference-only Bible reference detected."

    return True, "reference_with_text", "Bible reference plus surrounding text detected."

def append_note(existing, note: str) -> str:
    existing = "" if existing is None else str(existing).strip()
    if not existing:
        return note
    if note in existing:
        return existing
    return f"{existing} {note}"

classified = df_units["source_text"].map(classify_bible_reference_text)
df_units["contains_bible_reference"] = classified.map(lambda item: item[0])
df_units["bible_reference_category"] = classified.map(lambda item: item[1])
df_units["bible_reference_note"] = classified.map(lambda item: item[2])

test_reference_strings = [
    "Rev 14:6-13*",
    "Rev 13*",
    "Rev 14:6-13",
    "Rev 13",
]

for s in test_reference_strings:
    print(s, "->", classify_bible_reference_text(s)[1])

preserve_reference_mask = df_units["bible_reference_category"].isin([
    "cross_reference_only",
    "reference_only",
])

df_units.loc[preserve_reference_mask, "translation_action"] = "preserve"
df_units.loc[preserve_reference_mask, "target_text"] = df_units.loc[preserve_reference_mask, "source_text"]
df_units.loc[preserve_reference_mask, "translation_note"] = df_units.loc[
    preserve_reference_mask,
    "translation_note",
].map(lambda note: append_note(note, "Bible reference preserved unchanged."))

reference_only_mask = df_units["bible_reference_category"].eq("reference_only")
df_units.loc[reference_only_mask, "translation_note"] = "Bible reference preserved unchanged."

# Reference-plus-text categories are intentionally not preserved by this first-pass detector.
reference_text_mask = df_units["bible_reference_category"].isin([
    "leading_reference_with_text",
    "reference_with_text",
])
df_units.loc[reference_text_mask, "translation_action"] = "translate"

cf_mask = df_units["source_text"].apply(starts_with_cf)
df_units.loc[cf_mask, "contains_bible_reference"] = True
df_units.loc[cf_mask, "bible_reference_category"] = "cross_reference_only"
df_units.loc[cf_mask, "translation_action"] = "preserve"
df_units.loc[cf_mask, "target_text"] = df_units.loc[cf_mask, "source_text"]
df_units.loc[cf_mask, "bible_reference_note"] = "Cross-reference beginning with cf. detected."
df_units.loc[cf_mask, "translation_note"] = "Bible cross-reference preserved unchanged."

print("Rows beginning with cf.:", int(cf_mask.sum()))
print(
    "Rows beginning with cf. classified as cross_reference_only:",
    int(df_units.loc[cf_mask, "bible_reference_category"].eq("cross_reference_only").sum()),
)
display(
    df_units.loc[
        cf_mask,
        [
            "source_file",
            "group_stack",
            "source_text",
            "bible_reference_category",
            "translation_action",
            "translation_note",
        ],
    ].sort_values("source_text")
)

df_translate = df_units[df_units["translation_action"].eq("translate")].copy()

print("Total rows in df_units:", len(df_units))
print("Rows containing Bible references:", int(df_units["contains_bible_reference"].sum()))
print("Count by bible_reference_category:")
print(df_units["bible_reference_category"].value_counts(dropna=False))
print("Rows marked preserve:", int(df_units["translation_action"].eq("preserve").sum()))
print("Rows still marked translate:", len(df_translate))

review_cols = [
    "source_file",
    "group_stack",
    "source_text",
    "bible_reference_category",
    "translation_action",
    "translation_note",
]

print("Rows containing Bible references:")
display(df_units.loc[df_units["contains_bible_reference"], review_cols].copy())

print("Leading references with text:")
display(df_units.loc[df_units["bible_reference_category"].eq("leading_reference_with_text"), review_cols].copy())

print("Reference with text rows:")
display(df_units.loc[df_units["bible_reference_category"].eq("reference_with_text"), review_cols].copy())


Loaded Bible reference aliases from bible_reference_helpers: 242
Rev 14:6-13* -> reference_only
Rev 13* -> reference_only
Rev 14:6-13 -> reference_only
Rev 13 -> reference_only
Rows beginning with cf.: 43
Rows beginning with cf. classified as cross_reference_only: 43


,source_file,group_stack,source_text,bible_reference_category,translation_action,translation_note
245,StructureOfRevelation.svg,throne_room,"cf. 1 Ki 22,Is 6,Jer 17:12,Eze 1,3:12-14,Eze 1...",cross_reference_only,preserve,Bible cross-reference preserved unchanged.
122,StructureOfRevelation.svg,_x37__visions,"cf. Dan 7, Jer 15:2, 43:11",cross_reference_only,preserve,Bible cross-reference preserved unchanged.
228,StructureOfRevelation.svg,interlude,"cf. Dan 7,12, Mal 3:16",cross_reference_only,preserve,Bible cross-reference preserved unchanged.
240,StructureOfRevelation.svg,interlude,"cf. Dan 7.2, Eze 9:4",cross_reference_only,preserve,Bible cross-reference preserved unchanged.
66,StructureOfRevelation.svg,_x37__trumpets,"cf. Ex 15:23,Is 14:12,Jer 9:15",cross_reference_only,preserve,Bible cross-reference preserved unchanged.
74,StructureOfRevelation.svg,_x37__trumpets,"cf. Ex 7-12, Lev 25:8-9, Ps 47:5, Is 18:3, Is ...",cross_reference_only,preserve,Bible cross-reference preserved unchanged.
1,StructureOfRevelation.svg,_x37__bowls,"cf. Ex 7:17-21,Is 49:26,Rev 8:8-10,11:6",cross_reference_only,preserve,Bible cross-reference preserved unchanged.
0,StructureOfRevelation.svg,_x37__bowls,cf. Ex 9:9-11,cross_reference_only,preserve,Bible cross-reference preserved unchanged.
42,StructureOfRevelation.svg,_x37__seals,"cf. Eze 14:19-21,Mt 24:7-8",cross_reference_only,preserve,Bible cross-reference preserved unchanged.
69,StructureOfRevelation.svg,_x37__trumpets,cf. Eze 2:10,cross_reference_only,preserve,Bible cross-reference preserved unchanged.


Total rows in df_units: 249
Rows containing Bible references: 82
Count by bible_reference_category:
bible_reference_category
None                    167
cross_reference_only     43
reference_only           31
reference_with_text       8
Name: count, dtype: int64
Rows marked preserve: 74
Rows still marked translate: 175
Rows containing Bible references:


,source_file,group_stack,source_text,bible_reference_category,translation_action,translation_note
0,StructureOfRevelation.svg,_x37__bowls,cf. Ex 9:9-11,cross_reference_only,preserve,Bible cross-reference preserved unchanged.
1,StructureOfRevelation.svg,_x37__bowls,"cf. Ex 7:17-21,Is 49:26,Rev 8:8-10,11:6",cross_reference_only,preserve,Bible cross-reference preserved unchanged.
2,StructureOfRevelation.svg,_x37__bowls,cf. Rev 14:18,cross_reference_only,preserve,Bible cross-reference preserved unchanged.
3,StructureOfRevelation.svg,_x37__bowls,"cf. Rev 9:14,13:1",cross_reference_only,preserve,Bible cross-reference preserved unchanged.
6,StructureOfRevelation.svg,_x37__bowls,Rev 16:3,reference_only,preserve,Bible reference preserved unchanged.
...,...,...,...,...,...,...
239,StructureOfRevelation.svg,interlude,"cf. Isa 33; Jer 1Eze 1-3,7-12,14,37,40Dan 7,12...",cross_reference_only,preserve,Bible cross-reference preserved unchanged.
240,StructureOfRevelation.svg,interlude,"cf. Dan 7.2, Eze 9:4",cross_reference_only,preserve,Bible cross-reference preserved unchanged.
241,StructureOfRevelation.svg,interlude,cf. Eze 48,cross_reference_only,preserve,Bible cross-reference preserved unchanged.
244,StructureOfRevelation.svg,throne_room,After these things I looked and saw a door ope...,reference_with_text,translate,


Leading references with text:


,source_file,group_stack,source_text,bible_reference_category,translation_action,translation_note


Reference with text rows:


,source_file,group_stack,source_text,bible_reference_category,translation_action,translation_note
201,StructureOfRevelation.svg,future_glory,"There were lightnings, sounds, thunders; there...",reference_with_text,translate,
212,StructureOfRevelation.svg,future_glory,The Last Trumpet of 1 Cor 15:52,reference_with_text,translate,
213,StructureOfRevelation.svg,future_glory,"Out of the throne proceed lightnings, sounds,a...",reference_with_text,translate,
216,StructureOfRevelation.svg,future_glory,"Thunders, sounds, lightnings, andan earthquake...",reference_with_text,translate,
218,StructureOfRevelation.svg,future_glory,"Lightnings, sounds, thunders, an earth-quake, ...",reference_with_text,translate,
227,StructureOfRevelation.svg,interlude,If anyone was not found written in the book of...,reference_with_text,translate,
235,StructureOfRevelation.svg,interlude,"I looked and behold, a great multitude which n...",reference_with_text,translate,
244,StructureOfRevelation.svg,throne_room,After these things I looked and saw a door ope...,reference_with_text,translate,


In [8]:
for category in [
    "cross_reference_only",
    "reference_only",
    "leading_reference_with_text",
    "reference_with_text",
]:
    print("\n" + "=" * 80)
    print(category)
    print("=" * 80)
    display(
        df_units[df_units["bible_reference_category"].eq(category)][
            [
                "source_file",
                "group_stack",
                "source_text",
                "translation_action",
                "translation_note",
            ]
        ].head(15)
    )



cross_reference_only


,source_file,group_stack,source_text,translation_action,translation_note
0,StructureOfRevelation.svg,_x37__bowls,cf. Ex 9:9-11,preserve,Bible cross-reference preserved unchanged.
1,StructureOfRevelation.svg,_x37__bowls,"cf. Ex 7:17-21,Is 49:26,Rev 8:8-10,11:6",preserve,Bible cross-reference preserved unchanged.
2,StructureOfRevelation.svg,_x37__bowls,cf. Rev 14:18,preserve,Bible cross-reference preserved unchanged.
3,StructureOfRevelation.svg,_x37__bowls,"cf. Rev 9:14,13:1",preserve,Bible cross-reference preserved unchanged.
16,StructureOfRevelation.svg,_x37__bowls,"cf. Joe 2,Rev 6:12",preserve,Bible cross-reference preserved unchanged.
23,StructureOfRevelation.svg,_x37__churches,"cf. Is 41:4 & 44:6 & 48:12, Dan 7:13, Zec 4:2 ...",preserve,Bible cross-reference preserved unchanged.
38,StructureOfRevelation.svg,_x37__seals,"cf. Is 2,24, Eze 4-5, Jer 15-16 & 30, Dan 8:26...",preserve,Bible cross-reference preserved unchanged.
39,StructureOfRevelation.svg,_x37__seals,"cf. Ps 45,Mt 24:3-5",preserve,Bible cross-reference preserved unchanged.
40,StructureOfRevelation.svg,_x37__seals,cf. Mt 24:6,preserve,Bible cross-reference preserved unchanged.
41,StructureOfRevelation.svg,_x37__seals,cf. Mt 24:7,preserve,Bible cross-reference preserved unchanged.



reference_only


,source_file,group_stack,source_text,translation_action,translation_note
6,StructureOfRevelation.svg,_x37__bowls,Rev 16:3,preserve,Bible reference preserved unchanged.
8,StructureOfRevelation.svg,_x37__bowls,Rev 16:4-7,preserve,Bible reference preserved unchanged.
11,StructureOfRevelation.svg,_x37__bowls,Rev 16:10-11,preserve,Bible reference preserved unchanged.
17,StructureOfRevelation.svg,_x37__bowls,Rev 16:2,preserve,Bible reference preserved unchanged.
18,StructureOfRevelation.svg,_x37__bowls,Rev 16:8-9,preserve,Bible reference preserved unchanged.
19,StructureOfRevelation.svg,_x37__bowls,Rev 16:12-14,preserve,Bible reference preserved unchanged.
33,StructureOfRevelation.svg,_x37__seals,Rev 6:3-4,preserve,Bible reference preserved unchanged.
34,StructureOfRevelation.svg,_x37__seals,Rev 6:5-6,preserve,Bible reference preserved unchanged.
35,StructureOfRevelation.svg,_x37__seals,Rev 6:7-8,preserve,Bible reference preserved unchanged.
36,StructureOfRevelation.svg,_x37__seals,Rev 6:9-11,preserve,Bible reference preserved unchanged.



leading_reference_with_text


,source_file,group_stack,source_text,translation_action,translation_note



reference_with_text


,source_file,group_stack,source_text,translation_action,translation_note
201,StructureOfRevelation.svg,future_glory,"There were lightnings, sounds, thunders; there...",translate,
212,StructureOfRevelation.svg,future_glory,The Last Trumpet of 1 Cor 15:52,translate,
213,StructureOfRevelation.svg,future_glory,"Out of the throne proceed lightnings, sounds,a...",translate,
216,StructureOfRevelation.svg,future_glory,"Thunders, sounds, lightnings, andan earthquake...",translate,
218,StructureOfRevelation.svg,future_glory,"Lightnings, sounds, thunders, an earth-quake, ...",translate,
227,StructureOfRevelation.svg,interlude,If anyone was not found written in the book of...,translate,
235,StructureOfRevelation.svg,interlude,"I looked and behold, a great multitude which n...",translate,
244,StructureOfRevelation.svg,throne_room,After these things I looked and saw a door ope...,translate,


In [9]:
inspect_df = df_units.loc[df_units["contains_bible_reference"], review_cols].copy()
inspect_df = inspect_df.sort_values("bible_reference_category")
inspect_df.shape

(82, 6)

In [10]:
inspect_df['bible_reference_category'].value_counts()

bible_reference_category
cross_reference_only    43
reference_only          31
reference_with_text      8
Name: count, dtype: int64

In [11]:
# # cross_reference_only
# # reference_only
# # leading_reference_with_text
# # reference_with_text

# cat = 'cross_reference_only'
# display(inspect_df[inspect_df["bible_reference_category"].eq(cat)])


## Revelation-specific review: repeated source text


In [12]:
repeated_text_counts = (
    df_units["source_text"]
    .value_counts(dropna=False)
    .reset_index()
)

repeated_text_counts.columns = ["source_text", "count"]
repeated_text = repeated_text_counts[repeated_text_counts["count"] > 1].copy()

def summarize_translation_actions(actions):
    values = sorted({str(value) for value in actions.dropna() if str(value).strip()})
    if not values:
        return ""
    if len(values) == 1:
        return values[0]
    return "mixed"

def summarize_bible_reference_categories(categories):
    values = sorted({str(value) for value in categories.dropna() if str(value).strip()})
    return ", ".join(values)

translation_action_summary = (
    df_units.groupby("source_text", dropna=False)["translation_action"]
    .apply(summarize_translation_actions)
    .reset_index(name="translation_action_summary")
)

if "bible_reference_category" in df_units.columns:
    bible_reference_category_summary = (
        df_units.groupby("source_text", dropna=False)["bible_reference_category"]
        .apply(summarize_bible_reference_categories)
        .reset_index(name="bible_reference_categories")
    )
else:
    bible_reference_category_summary = repeated_text_counts[["source_text"]].copy()
    bible_reference_category_summary["bible_reference_categories"] = ""

repeated_text = repeated_text.merge(
    translation_action_summary,
    on="source_text",
    how="left",
)
repeated_text = repeated_text.merge(
    bible_reference_category_summary,
    on="source_text",
    how="left",
)
repeated_text = repeated_text.sort_values(
    ["count", "source_text"],
    ascending=[False, True],
).reset_index(drop=True)

print("Total unique source_text values:", len(repeated_text_counts))
print("Repeated source_text values:", len(repeated_text))
print("Total rows represented by repeated source_text values:", int(repeated_text["count"].sum()))
print("Top 20 repeated source_text values:")
display(repeated_text.head(20))

display(repeated_text)

repeated_source_values = set(repeated_text["source_text"])
repeated_text_review = df_units[df_units["source_text"].isin(repeated_source_values)].copy()
repeated_text_review = repeated_text_review.sort_values([
    "source_text",
    "source_file",
    "group_stack",
])

review_cols = [
    "source_file",
    "group_stack",
    "source_text",
    "translation_action",
    "bible_reference_category",
    "translation_note",
    "target_text",
]
review_cols = [col for col in review_cols if col in repeated_text_review.columns]

display(repeated_text_review[review_cols])



Total unique source_text values: 227
Repeated source_text values: 12
Total rows represented by repeated source_text values: 34
Top 20 repeated source_text values:


,source_text,count,translation_action_summary,bible_reference_categories
0,7:,7,translate,
1,th,4,translate,
2,*,3,translate,
3,C,3,translate,
4,Interlude:,3,translate,
5,49 cross-references,2,translate,
6,A’,2,translate,
7,B’,2,translate,
8,C’,2,translate,
9,D,2,translate,


,source_text,count,translation_action_summary,bible_reference_categories
0,7:,7,translate,
1,th,4,translate,
2,*,3,translate,
3,C,3,translate,
4,Interlude:,3,translate,
5,49 cross-references,2,translate,
6,A’,2,translate,
7,B’,2,translate,
8,C’,2,translate,
9,D,2,translate,


,source_file,group_stack,source_text,translation_action,bible_reference_category,translation_note,target_text
197,StructureOfRevelation.svg,endurance_asterisks,*,translate,None,,None
198,StructureOfRevelation.svg,endurance_asterisks,*,translate,None,,None
199,StructureOfRevelation.svg,endurance_asterisks,*,translate,None,,None
104,StructureOfRevelation.svg,_x37__victories_and_condemnations,49 cross-references,translate,None,,None
209,StructureOfRevelation.svg,future_glory,49 cross-references,translate,None,,None
150,StructureOfRevelation.svg,chiasm,7:,translate,None,,None
156,StructureOfRevelation.svg,chiasm,7:,translate,None,,None
164,StructureOfRevelation.svg,chiasm,7:,translate,None,,None
169,StructureOfRevelation.svg,chiasm,7:,translate,None,,None
174,StructureOfRevelation.svg,chiasm,7:,translate,None,,None


## Revelation-specific review: repeated phrase patterns


In [13]:
from collections import Counter
import re


def normalize_for_phrase_review(value):
    text = "" if value is None else str(value)
    return re.sub(r"\s+", " ", text.strip().lower())

source_text_normalized = df_units["source_text"].map(normalize_for_phrase_review)
cross_reference_phrase_mask = source_text_normalized.str.contains(
    r"\bcross-references?\b",
    case=False,
    regex=True,
    na=False,
)

cross_reference_phrase_review = df_units.loc[cross_reference_phrase_mask].copy()

print('Rows containing "cross-reference" or "cross-references":', len(cross_reference_phrase_review))
print("Exact source_text value counts among cross-reference phrase rows:")
display(cross_reference_phrase_review["source_text"].value_counts(dropna=False).rename_axis("source_text").reset_index(name="count"))

if "translation_action" in cross_reference_phrase_review.columns:
    print("translation_action counts among cross-reference phrase rows:")
    display(cross_reference_phrase_review["translation_action"].value_counts(dropna=False).rename_axis("translation_action").reset_index(name="count"))

if "bible_reference_category" in cross_reference_phrase_review.columns:
    print("bible_reference_category counts among cross-reference phrase rows:")
    display(cross_reference_phrase_review["bible_reference_category"].value_counts(dropna=False).rename_axis("bible_reference_category").reset_index(name="count"))

cross_reference_review_cols = [
    "source_file",
    "group_stack",
    "source_text",
    "translation_action",
    "bible_reference_category",
    "translation_note",
    "target_text",
]
cross_reference_review_cols = [col for col in cross_reference_review_cols if col in cross_reference_phrase_review.columns]
display(cross_reference_phrase_review[cross_reference_review_cols])

word_token_re = re.compile(r"[\w]+(?:[-'][\w]+)?", flags=re.UNICODE)

def tokenize_for_phrase_review(text):
    return word_token_re.findall(normalize_for_phrase_review(text))

token_counter = Counter()
bigram_counter = Counter()

for source_text in df_units["source_text"]:
    tokens = tokenize_for_phrase_review(source_text)
    meaningful_tokens = [
        token for token in tokens
        if len(token) >= 4 and not token.isnumeric()
    ]
    token_counter.update(meaningful_tokens)
    bigram_counter.update(
        " ".join(pair)
        for pair in zip(tokens, tokens[1:])
    )

token_frequency = pd.DataFrame(
    token_counter.most_common(50),
    columns=["token", "count"],
)

bigram_frequency = pd.DataFrame(
    [(bigram, count) for bigram, count in bigram_counter.most_common() if count > 1][:50],
    columns=["bigram", "count"],
)

print("Top 50 repeated meaningful tokens:")
display(token_frequency)

print("Top 50 repeated bigrams:")
display(bigram_frequency)




Rows containing "cross-reference" or "cross-references": 27
Exact source_text value counts among cross-reference phrase rows:


,source_text,count
0,49 cross-references,2
1,232 cross-references,1
2,113 cross-references,1
3,87 cross-references,1
4,33 cross-references,1
5,373 cross-references,1
6,311 cross-references,1
7,709 cross-references,1
8,153 cross-references,1
9,76 cross-references,1


translation_action counts among cross-reference phrase rows:


,translation_action,count
0,translate,27


bible_reference_category counts among cross-reference phrase rows:


...

,source_file,group_stack,source_text,translation_action,bible_reference_category,translation_note,target_text
4,StructureOfRevelation.svg,_x37__bowls,232 cross-references,translate,None,,None
24,StructureOfRevelation.svg,_x37__churches,1307 cross-references,translate,None,,None
46,StructureOfRevelation.svg,_x37__seals,443 cross-references,translate,None,,None
70,StructureOfRevelation.svg,_x37__trumpets,322 cross-references,translate,None,,None
97,StructureOfRevelation.svg,_x37__victories_and_condemnations,243 cross-references,translate,None,,None
98,StructureOfRevelation.svg,_x37__victories_and_condemnations,313 cross-references,translate,None,,None
99,StructureOfRevelation.svg,_x37__victories_and_condemnations,70 cross-references,translate,None,,None
100,StructureOfRevelation.svg,_x37__victories_and_condemnations,126 cross-references,translate,None,,None
101,StructureOfRevelation.svg,_x37__victories_and_condemnations,152 cross-references,translate,None,,None
103,StructureOfRevelation.svg,_x37__victories_and_condemnations,105 cross-references,translate,None,,None


Top 50 repeated meaningful tokens:


,token,count
0,cross-references,27
1,revelation,17
2,seven,8
3,great,4
4,interlude,4
5,lightnings,4
6,sounds,4
7,thunders,4
8,bowls,3
9,fire,3


Top 50 repeated bigrams:


,bigram,count
0,cf is,13
1,rev 16,8
2,rev 6,8
3,rev 8,7
4,cf rev,7
5,dan 7,7
6,mt 24,7
7,cf ex,4
8,rev 14,4
9,rev 9,4


## Revelation-specific translation notes: repeated terms


In [14]:
import re

if "repeated_phrase_note" not in df_units.columns:
    df_units["repeated_phrase_note"] = ""
else:
    df_units["repeated_phrase_note"] = df_units["repeated_phrase_note"].fillna("")

cross_reference_term_note = 'Repeated phrase detected: "cross-reference(s)". Translate consistently according to the target language; preserve any leading number unchanged.'
revelation_term_note = 'Repeated term detected: "Revelation". Translate or preserve according to the target-language prompt and context. If the term is part of a Bible reference, preserve or normalize according to the Bible-reference policy.'

def remove_old_repeated_phrase_notes(existing) -> str:
    existing = "" if existing is None else str(existing).strip()
    old_note_patterns = [
        r"Translate cross-reference/cross-references consistently as .*?; preserve any leading number unchanged\.?",
        r"Translate Revelation as .*? when it is ordinary text; preserve it unchanged when it is part of a Bible reference already marked preserve\.?",
    ]
    for old_note_pattern in old_note_patterns:
        existing = re.sub(old_note_pattern, "", existing)
    return " ".join(existing.split())

df_units["repeated_phrase_note"] = df_units["repeated_phrase_note"].map(remove_old_repeated_phrase_notes)

def append_repeated_phrase_note(existing, note: str) -> str:
    existing = "" if existing is None else str(existing).strip()
    if not existing:
        return note
    if note in existing:
        return existing
    return f"{existing} {note}"

source_text_for_repeated_notes = df_units["source_text"].fillna("").astype(str)
cross_reference_term_mask = source_text_for_repeated_notes.str.contains(
    r"\bcross-references?\b",
    case=False,
    regex=True,
    na=False,
)
revelation_term_mask = source_text_for_repeated_notes.str.contains(
    r"\bRevelation\b",
    case=False,
    regex=True,
    na=False,
)

df_units.loc[cross_reference_term_mask, "repeated_phrase_note"] = df_units.loc[
    cross_reference_term_mask,
    "repeated_phrase_note",
].map(lambda note: append_repeated_phrase_note(note, cross_reference_term_note))

df_units.loc[revelation_term_mask, "repeated_phrase_note"] = df_units.loc[
    revelation_term_mask,
    "repeated_phrase_note",
].map(lambda note: append_repeated_phrase_note(note, revelation_term_note))

print('Rows containing "cross-reference" or "cross-references":', int(cross_reference_term_mask.sum()))
print("translation_action counts for cross-reference phrase rows:")
print(df_units.loc[cross_reference_term_mask, "translation_action"].value_counts(dropna=False))

print('Rows containing "Revelation":', int(revelation_term_mask.sum()))
print("translation_action counts for Revelation phrase rows:")
print(df_units.loc[revelation_term_mask, "translation_action"].value_counts(dropna=False))

repeated_term_review_cols = [
    "source_file",
    "group_stack",
    "source_text",
    "translation_action",
    "bible_reference_category",
    "repeated_phrase_note",
    "translation_note",
    "target_text",
]
repeated_term_review_cols = [col for col in repeated_term_review_cols if col in df_units.columns]

print("Cross-reference phrase rows:")
display(df_units.loc[cross_reference_term_mask, repeated_term_review_cols].copy())

print("Revelation phrase rows:")
display(df_units.loc[revelation_term_mask, repeated_term_review_cols].copy())

df_translate = df_units[df_units["translation_action"].eq("translate")].copy()






Rows containing "cross-reference" or "cross-references": 27
translation_action counts for cross-reference phrase rows:
translation_action
translate    27
Name: count, dtype: int64
Rows containing "Revelation": 17
translation_action counts for Revelation phrase rows:
translation_action
translate    17
Name: count, dtype: int64
Cross-reference phrase rows:


,source_file,group_stack,source_text,translation_action,bible_reference_category,repeated_phrase_note,translation_note,target_text
4,StructureOfRevelation.svg,_x37__bowls,232 cross-references,translate,None,"Repeated phrase detected: ""cross-reference(s)""...",,None
24,StructureOfRevelation.svg,_x37__churches,1307 cross-references,translate,None,"Repeated phrase detected: ""cross-reference(s)""...",,None
46,StructureOfRevelation.svg,_x37__seals,443 cross-references,translate,None,"Repeated phrase detected: ""cross-reference(s)""...",,None
70,StructureOfRevelation.svg,_x37__trumpets,322 cross-references,translate,None,"Repeated phrase detected: ""cross-reference(s)""...",,None
97,StructureOfRevelation.svg,_x37__victories_and_condemnations,243 cross-references,translate,None,"Repeated phrase detected: ""cross-reference(s)""...",,None
98,StructureOfRevelation.svg,_x37__victories_and_condemnations,313 cross-references,translate,None,"Repeated phrase detected: ""cross-reference(s)""...",,None
99,StructureOfRevelation.svg,_x37__victories_and_condemnations,70 cross-references,translate,None,"Repeated phrase detected: ""cross-reference(s)""...",,None
100,StructureOfRevelation.svg,_x37__victories_and_condemnations,126 cross-references,translate,None,"Repeated phrase detected: ""cross-reference(s)""...",,None
101,StructureOfRevelation.svg,_x37__victories_and_condemnations,152 cross-references,translate,None,"Repeated phrase detected: ""cross-reference(s)""...",,None
103,StructureOfRevelation.svg,_x37__victories_and_condemnations,105 cross-references,translate,None,"Repeated phrase detected: ""cross-reference(s)""...",,None


Revelation phrase rows:


,source_file,group_stack,source_text,translation_action,bible_reference_category,repeated_phrase_note,translation_note,target_text
12,StructureOfRevelation.svg,_x37__bowls,Revelation 16:1-16,translate,None,"Repeated term detected: ""Revelation"". Translat...",,None
21,StructureOfRevelation.svg,_x37__churches,Revelation 1-3,translate,None,"Repeated term detected: ""Revelation"". Translat...",,None
22,StructureOfRevelation.svg,_x37__churches,see 7 Churches of Revelation graphic,translate,None,"Repeated term detected: ""Revelation"". Translat...",,None
47,StructureOfRevelation.svg,_x37__seals,Revelation 5-6,translate,None,"Repeated term detected: ""Revelation"". Translat...",,None
73,StructureOfRevelation.svg,_x37__trumpets,Revelation 8.6 - 9.21,translate,None,"Repeated term detected: ""Revelation"". Translat...",,None
96,StructureOfRevelation.svg,_x37__victories_and_condemnations,Revelation 20:7-10,translate,None,"Repeated term detected: ""Revelation"". Translat...",,None
118,StructureOfRevelation.svg,_x37__visions,Revelation 15,translate,None,"Repeated term detected: ""Revelation"". Translat...",,None
203,StructureOfRevelation.svg,future_glory,Revelation 21-22,translate,None,"Repeated term detected: ""Revelation"". Translat...",,None
205,StructureOfRevelation.svg,future_glory,Revelation 8:1-5,translate,None,"Repeated term detected: ""Revelation"". Translat...",,None
206,StructureOfRevelation.svg,future_glory,Revelation 11:15-19,translate,None,"Repeated term detected: ""Revelation"". Translat...",,None


In [15]:
df_units["translation_action"].value_counts(dropna=False)

translation_action
translate    175
preserve      74
Name: count, dtype: int64

In [16]:
df_units.loc[
    df_units["translation_action"].isin(["preserve", "manual"]) &
    df_units["target_text"].isna(),
    ["source_text", "translation_action", "target_text"]
]

,source_text,translation_action,target_text


In [17]:
df_units.loc[
    df_units["translation_action"].eq("translate") &
    df_units["target_text"].notna(),
    ["source_text", "translation_action", "target_text"]
]

,source_text,translation_action,target_text


In [18]:
df_units[df_units["translation_note"].str.contains("Greek", na=False)][
    ["source_text", "translation_action", "target_text", "translation_note"]
]

,source_text,translation_action,target_text,translation_note


In [19]:
df_units["bible_reference_category"].value_counts(dropna=False)

bible_reference_category
None                    167
cross_reference_only     43
reference_only           31
reference_with_text       8
Name: count, dtype: int64

In [20]:
pd.crosstab(
    df_units["bible_reference_category"].fillna("None"),
    df_units["translation_action"].fillna("MISSING")
)

translation_action,preserve,translate
bible_reference_category,,
None,0,167
cross_reference_only,43,0
reference_only,31,0
reference_with_text,0,8


In [21]:
df_translate = df_units[df_units["translation_action"].eq("translate")].copy()

print("Rows in df_units:", len(df_units))
print("Rows to translate:", len(df_translate))
print("Rows preserved/manual:", len(df_units) - len(df_translate))

Rows in df_units: 249
Rows to translate: 175
Rows preserved/manual: 74


In [22]:
display(
    df_translate[
        [
            "source_file",
            "group_stack",
            "source_text",
            "translation_action",
            "bible_reference_category",
            "repeated_phrase_note",
            "translation_note",
        ]
    ].head(50)
)

,source_file,group_stack,source_text,translation_action,bible_reference_category,repeated_phrase_note,translation_note
4,StructureOfRevelation.svg,_x37__bowls,232 cross-references,translate,None,"Repeated phrase detected: ""cross-reference(s)""...",
5,StructureOfRevelation.svg,_x37__bowls,All life insea dies,translate,None,,
7,StructureOfRevelation.svg,_x37__bowls,Water toblood,translate,None,,
9,StructureOfRevelation.svg,_x37__bowls,Darkness,translate,None,,
10,StructureOfRevelation.svg,_x37__bowls,7 Bowls,translate,None,,
12,StructureOfRevelation.svg,_x37__bowls,Revelation 16:1-16,translate,None,"Repeated term detected: ""Revelation"". Translat...",
13,StructureOfRevelation.svg,_x37__bowls,Sores,translate,None,,
14,StructureOfRevelation.svg,_x37__bowls,Fire from Sun,translate,None,,
15,StructureOfRevelation.svg,_x37__bowls,Euphratesdries,translate,None,,
20,StructureOfRevelation.svg,_x37__churches,7 Churches,translate,None,,


In [23]:
# Preserved/manual rows should have target_text
display(
    df_units.loc[
        df_units["translation_action"].isin(["preserve", "manual"])
        & df_units["target_text"].isna(),
        ["source_text", "translation_action", "target_text", "translation_note"]
    ]
)

,source_text,translation_action,target_text,translation_note


In [24]:
# 3. Translate rows should usually not have target_text yet
display(
    df_units.loc[
        df_units["translation_action"].eq("translate")
        & df_units["target_text"].notna(),
        ["source_text", "translation_action", "target_text", "translation_note"]
    ]
)

,source_text,translation_action,target_text,translation_note


In [25]:
# Inspect preserved rows directly
display(
    df_units.loc[
        df_units["translation_action"].eq("preserve"),
        [
            "source_file",
            "group_stack",
            "source_text",
            "translation_action",
            "bible_reference_category",
            "translation_note",
            "target_text",
        ]
    ].head(100)
)

,source_file,group_stack,source_text,translation_action,bible_reference_category,translation_note,target_text
0,StructureOfRevelation.svg,_x37__bowls,cf. Ex 9:9-11,preserve,cross_reference_only,Bible cross-reference preserved unchanged.,cf. Ex 9:9-11
1,StructureOfRevelation.svg,_x37__bowls,"cf. Ex 7:17-21,Is 49:26,Rev 8:8-10,11:6",preserve,cross_reference_only,Bible cross-reference preserved unchanged.,"cf. Ex 7:17-21,Is 49:26,Rev 8:8-10,11:6"
2,StructureOfRevelation.svg,_x37__bowls,cf. Rev 14:18,preserve,cross_reference_only,Bible cross-reference preserved unchanged.,cf. Rev 14:18
3,StructureOfRevelation.svg,_x37__bowls,"cf. Rev 9:14,13:1",preserve,cross_reference_only,Bible cross-reference preserved unchanged.,"cf. Rev 9:14,13:1"
6,StructureOfRevelation.svg,_x37__bowls,Rev 16:3,preserve,reference_only,Bible reference preserved unchanged.,Rev 16:3
...,...,...,...,...,...,...,...
228,StructureOfRevelation.svg,interlude,"cf. Dan 7,12, Mal 3:16",preserve,cross_reference_only,Bible cross-reference preserved unchanged.,"cf. Dan 7,12, Mal 3:16"
239,StructureOfRevelation.svg,interlude,"cf. Isa 33; Jer 1Eze 1-3,7-12,14,37,40Dan 7,12...",preserve,cross_reference_only,Bible cross-reference preserved unchanged.,"cf. Isa 33; Jer 1Eze 1-3,7-12,14,37,40Dan 7,12..."
240,StructureOfRevelation.svg,interlude,"cf. Dan 7.2, Eze 9:4",preserve,cross_reference_only,Bible cross-reference preserved unchanged.,"cf. Dan 7.2, Eze 9:4"
241,StructureOfRevelation.svg,interlude,cf. Eze 48,preserve,cross_reference_only,Bible cross-reference preserved unchanged.,cf. Eze 48


In [26]:
display(
    df_units.loc[
        df_units["source_text"].astype(str).str.fullmatch(r"\W+", na=False),
        [
            "source_file",
            "group_stack",
            "source_text",
            "translation_action",
            "target_text",
        ],
    ]
)

,source_file,group_stack,source_text,translation_action,target_text
196,StructureOfRevelation.svg,endurance_asterisks,**,translate,None
197,StructureOfRevelation.svg,endurance_asterisks,*,translate,None
198,StructureOfRevelation.svg,endurance_asterisks,*,translate,None
199,StructureOfRevelation.svg,endurance_asterisks,*,translate,None


## Revelation-specific text filtering: preserve symbol-only text


In [27]:
def is_symbol_only_text(text):
    if text is None:
        return False
    try:
        if text != text:  # NaN is not equal to itself.
            return False
    except TypeError:
        pass

    text = str(text).strip()
    if not text:
        return False

    return not any(ch.isalpha() or ch.isdigit() for ch in text)

symbol_only_mask = df_units["source_text"].apply(is_symbol_only_text)

df_units.loc[symbol_only_mask, "translation_action"] = "preserve"
df_units.loc[symbol_only_mask, "target_text"] = df_units.loc[symbol_only_mask, "source_text"]
df_units.loc[symbol_only_mask, "translation_note"] = "Symbol-only text preserved unchanged."

print("Symbol-only rows detected:", int(symbol_only_mask.sum()))
print(
    "Symbol-only rows marked preserve:",
    int(df_units.loc[symbol_only_mask, "translation_action"].eq("preserve").sum()),
)

symbol_only_review_cols = [
    "source_file",
    "group_stack",
    "source_text",
    "translation_action",
    "translation_note",
    "target_text",
]
symbol_only_review_cols = [col for col in symbol_only_review_cols if col in df_units.columns]

display(df_units.loc[symbol_only_mask, symbol_only_review_cols].copy())

df_translate = df_units[df_units["translation_action"].eq("translate")].copy()



Symbol-only rows detected: 4
Symbol-only rows marked preserve: 4


,source_file,group_stack,source_text,translation_action,translation_note,target_text
196,StructureOfRevelation.svg,endurance_asterisks,**,preserve,Symbol-only text preserved unchanged.,**
197,StructureOfRevelation.svg,endurance_asterisks,*,preserve,Symbol-only text preserved unchanged.,*
198,StructureOfRevelation.svg,endurance_asterisks,*,preserve,Symbol-only text preserved unchanged.,*
199,StructureOfRevelation.svg,endurance_asterisks,*,preserve,Symbol-only text preserved unchanged.,*


## Revelation-specific text filtering: preserve ordinal suffix fragments

In [28]:
def is_ordinal_suffix_fragment(text):
    if text is None:
        return False
    try:
        if pd.isna(text):
            return False
    except (TypeError, ValueError):
        pass

    text = str(text).strip()
    return text.lower() in {"st", "nd", "rd", "th"}

ordinal_suffix_mask = df_units["source_text"].apply(is_ordinal_suffix_fragment)

df_units.loc[ordinal_suffix_mask, "translation_action"] = "preserve"
df_units.loc[ordinal_suffix_mask, "target_text"] = df_units.loc[ordinal_suffix_mask, "source_text"]
df_units.loc[ordinal_suffix_mask, "translation_note"] = "Ordinal suffix fragment preserved unchanged."

print("Ordinal suffix fragments detected:", int(ordinal_suffix_mask.sum()))
print(
    "Ordinal suffix fragments marked preserve:",
    int(df_units.loc[ordinal_suffix_mask, "translation_action"].eq("preserve").sum()),
)

ordinal_suffix_review_cols = [
    "source_file",
    "group_stack",
    "source_text",
    "translation_action",
    "translation_note",
    "target_text",
]

display(df_units.loc[ordinal_suffix_mask, ordinal_suffix_review_cols].copy())

df_translate = df_units[df_units["translation_action"].eq("translate")].copy()


Ordinal suffix fragments detected: 7
Ordinal suffix fragments marked preserve: 7


,source_file,group_stack,source_text,translation_action,translation_note,target_text
149,StructureOfRevelation.svg,chiasm,nd,preserve,Ordinal suffix fragment preserved unchanged.,nd
155,StructureOfRevelation.svg,chiasm,rd,preserve,Ordinal suffix fragment preserved unchanged.,rd
162,StructureOfRevelation.svg,chiasm,th,preserve,Ordinal suffix fragment preserved unchanged.,th
168,StructureOfRevelation.svg,chiasm,th,preserve,Ordinal suffix fragment preserved unchanged.,th
173,StructureOfRevelation.svg,chiasm,th,preserve,Ordinal suffix fragment preserved unchanged.,th
178,StructureOfRevelation.svg,chiasm,th,preserve,Ordinal suffix fragment preserved unchanged.,th
194,StructureOfRevelation.svg,chiasm,st,preserve,Ordinal suffix fragment preserved unchanged.,st


In [29]:
print("Duplicate unit_keys:", df_translate["unit_key"].duplicated().sum())

Duplicate unit_keys: 0


### Parse to JSON and save to JSON files directory
**CAUTION:** this will overwrite any previously run/saved versions

In [30]:
import json
from pathlib import Path

out_units = JSON_DIR / "translation_units.json"

if out_units.exists():
    print("Overwriting existing file:", out_units.relative_to(PROJECT_ROOT.parent))
else:
    print("Creating new file:", out_units.relative_to(PROJECT_ROOT.parent))

chars_written = out_units.write_text(
    json.dumps(df_translate.to_dict(orient="records"), ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Characters written:", chars_written)

Overwriting existing file: rev\json_files\translation_units.json
Characters written: 101595


In [31]:
df_all.head(2)

,source_file,text_id,group_stack,group_depth,text_raw,text_norm,has_tspans,tspan_count,x,y,transform,class,style,style_font_family,style_font_size,style_text_anchor,element_path,tspans
0,StructureOfRevelation.svg,None,_x37__bowls,1,cf. Ex 9:9-11,cf. Ex 9:9-11,False,0,None,None,matrix(1 0 0 1 253.9944 641.1695),None,None,None,None,None,svg/g#_x37__bowls/text[10],[]
1,StructureOfRevelation.svg,None,_x37__bowls,1,"cf. Ex 7:17-21,Is 49:26,Rev 8:8-10,11:6","cf. Ex 7:17-21,Is 49:26,Rev 8:8-10,11:6",False,0,None,None,matrix(1 0 0 1 372.5591 641.1695),None,None,None,None,None,svg/g#_x37__bowls/text[11],[]


In [32]:
df_units.head(4)

,unit_key,unit_type,source_file,group_stack,element_path,text_id,tspan_id,tspan_idx,source_text,translation_action,translation_note,target_text,contains_bible_reference,bible_reference_category,bible_reference_note,repeated_phrase_note
0,65971727bf7e86f3fecbb33b07be3acc2892de35,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[10],None,None,None,cf. Ex 9:9-11,preserve,Bible cross-reference preserved unchanged.,cf. Ex 9:9-11,True,cross_reference_only,Cross-reference beginning with cf. detected.,
1,8140f9ef3ce81a748de1998b176b8d1c1c072cd5,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[11],None,None,None,"cf. Ex 7:17-21,Is 49:26,Rev 8:8-10,11:6",preserve,Bible cross-reference preserved unchanged.,"cf. Ex 7:17-21,Is 49:26,Rev 8:8-10,11:6",True,cross_reference_only,Cross-reference beginning with cf. detected.,
2,69c90ab68fd7372766dc12d6f01b35d3857127b6,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[12],None,None,None,cf. Rev 14:18,preserve,Bible cross-reference preserved unchanged.,cf. Rev 14:18,True,cross_reference_only,Cross-reference beginning with cf. detected.,
3,3bf7cce880b6e43304712659ba23c60caf957731,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[13],None,None,None,"cf. Rev 9:14,13:1",preserve,Bible cross-reference preserved unchanged.,"cf. Rev 9:14,13:1",True,cross_reference_only,Cross-reference beginning with cf. detected.,


In [33]:
cols = [
    "unit_type",
    "source_file",
    "group_stack",
    "element_path",
    "text_id",
    "tspan_id",
    "tspan_idx",
    "source_text",
    "translation_action",
    "translation_note",
    "target_text",
]

df_units.loc[0:3, cols]

,unit_type,source_file,group_stack,element_path,text_id,tspan_id,tspan_idx,source_text,translation_action,translation_note,target_text
0,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[10],None,None,None,cf. Ex 9:9-11,preserve,Bible cross-reference preserved unchanged.,cf. Ex 9:9-11
1,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[11],None,None,None,"cf. Ex 7:17-21,Is 49:26,Rev 8:8-10,11:6",preserve,Bible cross-reference preserved unchanged.,"cf. Ex 7:17-21,Is 49:26,Rev 8:8-10,11:6"
2,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[12],None,None,None,cf. Rev 14:18,preserve,Bible cross-reference preserved unchanged.,cf. Rev 14:18
3,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[13],None,None,None,"cf. Rev 9:14,13:1",preserve,Bible cross-reference preserved unchanged.,"cf. Rev 9:14,13:1"


In [34]:
df_units.loc[0:3, ["element_path", "tspan_idx", "source_text"]].to_string(index=False)

'              element_path tspan_idx                             source_text\nsvg/g#_x37__bowls/text[10]      None                           cf. Ex 9:9-11\nsvg/g#_x37__bowls/text[11]      None cf. Ex 7:17-21,Is 49:26,Rev 8:8-10,11:6\nsvg/g#_x37__bowls/text[12]      None                           cf. Rev 14:18\nsvg/g#_x37__bowls/text[13]      None                       cf. Rev 9:14,13:1'

In [35]:
(
    df_units
    .sort_values(["source_file", "element_path", "tspan_idx"])
    .groupby(["source_file", "element_path"], dropna=False)["source_text"]
    .apply(lambda parts: "".join(str(x) for x in parts if str(x) != "nan"))
    .reset_index(name="combined_text")
    .head(20)
)

,source_file,element_path,combined_text
0,StructureOfRevelation.svg,svg/g#_x37__bowls/text[10],cf. Ex 9:9-11
1,StructureOfRevelation.svg,svg/g#_x37__bowls/text[11],"cf. Ex 7:17-21,Is 49:26,Rev 8:8-10,11:6"
2,StructureOfRevelation.svg,svg/g#_x37__bowls/text[12],cf. Rev 14:18
3,StructureOfRevelation.svg,svg/g#_x37__bowls/text[13],"cf. Rev 9:14,13:1"
4,StructureOfRevelation.svg,svg/g#_x37__bowls/text[14],232 cross-references
5,StructureOfRevelation.svg,svg/g#_x37__bowls/text[15],All life insea dies
6,StructureOfRevelation.svg,svg/g#_x37__bowls/text[16],Rev 16:3
7,StructureOfRevelation.svg,svg/g#_x37__bowls/text[17],Water toblood
8,StructureOfRevelation.svg,svg/g#_x37__bowls/text[18],Rev 16:4-7
9,StructureOfRevelation.svg,svg/g#_x37__bowls/text[19],Darkness


In [36]:
print("Total units:", len(df_units))
print("Rows marked preserve:", df_units["translation_action"].eq("preserve").sum())
print("Translate candidates:", df_units["translation_action"].eq("translate").sum())

df_units.head(20)

Total units: 249
Rows marked preserve: 85
Translate candidates: 164


,unit_key,unit_type,source_file,group_stack,element_path,text_id,tspan_id,tspan_idx,source_text,translation_action,translation_note,target_text,contains_bible_reference,bible_reference_category,bible_reference_note,repeated_phrase_note
0,65971727bf7e86f3fecbb33b07be3acc2892de35,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[10],None,None,None,cf. Ex 9:9-11,preserve,Bible cross-reference preserved unchanged.,cf. Ex 9:9-11,True,cross_reference_only,Cross-reference beginning with cf. detected.,
1,8140f9ef3ce81a748de1998b176b8d1c1c072cd5,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[11],None,None,None,"cf. Ex 7:17-21,Is 49:26,Rev 8:8-10,11:6",preserve,Bible cross-reference preserved unchanged.,"cf. Ex 7:17-21,Is 49:26,Rev 8:8-10,11:6",True,cross_reference_only,Cross-reference beginning with cf. detected.,
2,69c90ab68fd7372766dc12d6f01b35d3857127b6,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[12],None,None,None,cf. Rev 14:18,preserve,Bible cross-reference preserved unchanged.,cf. Rev 14:18,True,cross_reference_only,Cross-reference beginning with cf. detected.,
3,3bf7cce880b6e43304712659ba23c60caf957731,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[13],None,None,None,"cf. Rev 9:14,13:1",preserve,Bible cross-reference preserved unchanged.,"cf. Rev 9:14,13:1",True,cross_reference_only,Cross-reference beginning with cf. detected.,
4,4041ecaccb42dee3d7f29c75e00b137f86bf10cc,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[14],None,None,None,232 cross-references,translate,,None,False,None,,"Repeated phrase detected: ""cross-reference(s)""..."
5,6bb999b7f7c222c530baaa3de7036f1e146d9037,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[15],None,None,None,All life insea dies,translate,,None,False,None,,
6,3d375b2762a52fb3217330358fc98ebcbc81aacb,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[16],None,None,None,Rev 16:3,preserve,Bible reference preserved unchanged.,Rev 16:3,True,reference_only,Reference-only Bible reference detected.,
7,2e46ca8ff13d19e17a83d5683f948b0b18842b2d,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[17],None,None,None,Water toblood,translate,,None,False,None,,
8,2e8bdd1d58d4eb4202881730f7fffe13fd88c730,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[18],None,None,None,Rev 16:4-7,preserve,Bible reference preserved unchanged.,Rev 16:4-7,True,reference_only,Reference-only Bible reference detected.,
9,fc6ae5198a7f6b72c6c870f6aae0f115be396976,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[19],None,None,None,Darkness,translate,,None,False,None,,


In [37]:
# from pathlib import Path
# from datetime import datetime
# import json

# full_units_path = Path("json_files") / f"translation_units_full_prepared_{datetime.now():%Y%m%d_%H%M}.json"

# records = df_units.to_dict(orient="records")

# with full_units_path.open("w", encoding="utf-8") as f:
#     json.dump(records, f, ensure_ascii=False, indent=2)

# print("Saved full prepared units:", full_units_path)
# print("Rows:", len(records))

# print("\ntranslation_action counts:")
# print(df_units["translation_action"].value_counts(dropna=False))

# print("\nbible_reference_category counts:")
# print(df_units["bible_reference_category"].value_counts(dropna=False))

In [38]:
from pathlib import Path
from datetime import datetime
import json
import shutil

JSON_DIR = Path("json_files")
JSON_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M")

timestamped_path = JSON_DIR / f"translation_units_full_prepared_{timestamp}.json"
latest_path = JSON_DIR / "translation_units_full_prepared.json"

records = df_units.to_dict(orient="records")

with timestamped_path.open("w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

shutil.copyfile(timestamped_path, latest_path)

print("Saved timestamped full prepared units:", timestamped_path)
print("Saved latest full prepared units:", latest_path)
print("Rows:", len(records))

print("\ntranslation_action counts:")
print(df_units["translation_action"].value_counts(dropna=False))

print("\nbible_reference_category counts:")
print(df_units["bible_reference_category"].value_counts(dropna=False))

Saved timestamped full prepared units: json_files\translation_units_full_prepared_20260520_2013.json
Saved latest full prepared units: json_files\translation_units_full_prepared.json
Rows: 249

translation_action counts:
translation_action
translate    164
preserve      85
Name: count, dtype: int64

bible_reference_category counts:
bible_reference_category
None                    167
cross_reference_only     43
reference_only           31
reference_with_text       8
Name: count, dtype: int64
